In [56]:
## Load my files ##
import sys
sys.path.append('..')
from utils import get_sequence

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler
from torch import from_numpy as tnsr
from scipy.stats import bernoulli
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist as dist
from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import find_peaks
from scipy.spatial.distance import cosine
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import torch.nn.functional as F

In [57]:
n_community = 2
n_members = 3

tokens = []

for ii in range(n_community*n_members+1):
    tokens.append(
        chr(ord('A')+ii)
    )

In [ ]:
class brain(nn.Module):
    def __init__(self, input_size, hidden_wake_size, hidden_sleep_size, sleep_input_size, sleep_output_size, num_layers=1, num_layers_sleep=1):
        super(brain, self).__init__()

        self.rnn = nn.RNN(input_size+sleep_output_size, hidden_wake_size, num_layers, nonlinearity='relu', batch_first=True)
        self.wake_fc = nn.Linear(hidden_wake_size, len(tokens))
        self.context_fc = nn.Linear(hidden_wake_size, sleep_input_size)

        self.sleep_rnn = nn.RNN(sleep_input_size, hidden_sleep_size, num_layers_sleep, nonlinearity='relu', batch_first=True)
        self.sleep_out = nn.Linear(hidden_sleep_size, hidden_wake_size)
        self.sleep_fc = nn.Linear(hidden_sleep_size, sleep_output_size)

    def wake(self):
        self.rnn.requires_grad = True 
        self.wake_fc.requires_grad = True
        self.context_fc.requires_grad = True

        self.sleep_fc.requires_grad = True 

        self.sleep_rnn.requires_grad = False
        self.sleep_out.requires_grad = False

    def sleep(self):
        self.rnn.requires_grad = False 
        self.wake_fc.requires_grad = False
        self.context_fc.requires_grad = False 

        self.sleep_fc.requires_grad = False 

        self.sleep_rnn.requires_grad = True
        self.sleep_out.requires_grad = True
        
    def zero_sleep_weights(self):
        for name, param in self.sleep_rnn.named_parameters():
            nn.init.zeros_(param) 

    def forward(self, x, context, hw=None, hs=None):
        out, hs = self.sleep_rnn(context, hs)
        sleep_out = self.sleep_out(out[:,-1,:])
        context_out = self.sleep_fc(out)

        x = torch.cat((x,context_out), dim=2)
        
        out, hw = self.rnn(x, hw)

        context = self.context_fc(hw)
        out = self.wake_fc(out[:,-1,:])


        return out, sleep_out, context, hw, hs
 


In [59]:
class Dataset_converter(Dataset):
    def __init__(self, data, working_memory=1, short_term_memory=8):
        
        one_hot_encoded = np.zeros((len(data), len(tokens)), dtype=float)
        for ii, token in enumerate(data):
            one_hot_encoded[ii,ord(token)-65] = 1
        
        self.X = np.zeros((((len(data)-working_memory-short_term_memory)), short_term_memory, len(tokens)*working_memory))
        self.y = np.zeros((((len(data)-working_memory-short_term_memory)), len(tokens)))

        for ii in range(self.X.shape[0]):
            for jj in range(self.X.shape[1]):
                for kk in range(working_memory):
                    self.X[ii,jj,kk*len(tokens):(kk+1)*len(tokens)] = \
                    one_hot_encoded[ii+jj+kk,:]
                    
            self.y[ii] = \
                one_hot_encoded[ii+jj+kk+1,:]

        self.X = tnsr(self.X).float()
        self.y = tnsr(self.y).float()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [65]:
### initial training ###
total_samples = 50000
working_memory = 1
short_term_memory = 1
hidden_wake_size = 40
hidden_sleep_size = 10
sleep_input_size = 10
sleep_output_size = 5
num_layers_wake = 1
num_layers_sleep = 1
output_sleep = len(tokens)
input_size = len(tokens)*working_memory
lr = 1e-4
test_acc = []


c = []
context = torch.zeros(sleep_input_size,dtype=torch.float32)
data = get_sequence(total_samples, n_community, n_members, train_percent=1.0)#gen_seq(total_samples) #

data_set = Dataset_converter(data, working_memory, short_term_memory)
train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

network1 = brain(input_size, hidden_wake_size, hidden_sleep_size, sleep_input_size, sleep_output_size, num_layers_wake, num_layers_sleep)
# network1.zero_sleep_weights()
network1.wake()

optimizer = torch.optim.SGD(network1.parameters(), lr=lr, momentum=0.95)
criterion = torch.nn.CrossEntropyLoss()

total = 0
correct = np.zeros(1000,dtype=float)
for X, y in train_loader:
    optimizer.zero_grad()

    if total == 0:
        predicted_y, _, context_, hw, hs = network1(X, context.reshape(1,1,-1))
    else:
        predicted_y, _, context_, hw, hs = network1(X, context.reshape(1,1,-1), hw=mem, hs=mem_)
    
    # print(predicted_y.shape, y.shape)
    loss = criterion(predicted_y, y)
    #loss_repel = 1e-1 * network1.repulsion_loss(margin=1.0)
    
    # loss = loss #+ loss_repel
    
    loss.backward(retain_graph=True)
    optimizer.step()

    with torch.no_grad():
        mem=hw.clone()
        mem_=hs.clone()
        context = context_.clone()
        c.append(context)
        
        true_y = y.argmax(axis=1)
        estimated_y = predicted_y.argmax(axis=1)

        total += 1
        if true_y == estimated_y:
                correct[total%1000] = 1
        else:
            correct[total%1000] = 0

        test_acc.append(
            np.sum(correct)/total if total<1000 else np.sum(correct)/1000
        )
        if total%1000 == 0:
            print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc[-1]:.4f}')


Iter : 1001, loss: 1.9927, accuracy: 0.2370
Iter : 2001, loss: 1.9896, accuracy: 0.2500
Iter : 3001, loss: 2.0631, accuracy: 0.2500
Iter : 4001, loss: 2.0887, accuracy: 0.2500
Iter : 5001, loss: 2.0449, accuracy: 0.2500
Iter : 6001, loss: 1.9675, accuracy: 0.2620
Iter : 7001, loss: 1.9590, accuracy: 0.2680
Iter : 8001, loss: 1.7537, accuracy: 0.2910
Iter : 9001, loss: 2.0269, accuracy: 0.3510
Iter : 10001, loss: 2.0943, accuracy: 0.4170
Iter : 11001, loss: 1.7100, accuracy: 0.4640
Iter : 12001, loss: 1.6590, accuracy: 0.5260
Iter : 13001, loss: 1.6519, accuracy: 0.5760
Iter : 14001, loss: 1.6877, accuracy: 0.5960
Iter : 15001, loss: 1.9386, accuracy: 0.6290
Iter : 16001, loss: 2.0502, accuracy: 0.5940
Iter : 17001, loss: 1.5427, accuracy: 0.6440
Iter : 18001, loss: 2.0970, accuracy: 0.6420
Iter : 19001, loss: 1.5628, accuracy: 0.6460
Iter : 20001, loss: 2.0090, accuracy: 0.6560
Iter : 21001, loss: 1.2061, accuracy: 0.6690
Iter : 22001, loss: 1.5759, accuracy: 0.6580
Iter : 23001, loss:

In [66]:
data[-20:]

'EFDGBACGEFDGBCAGFDEG'

In [67]:
hs

tensor([[[0.3087, 0.1337, 0.7015, 0.0000, 0.3197, 0.0655, 0.2593, 0.6866,
          0.0000, 0.0735]]], grad_fn=<StackBackward0>)

In [68]:
for ii in range(20):
    print(c[-ii], data[-ii])

tensor([[[-0.0008, -0.0716, -0.1749,  0.2651, -0.1282,  0.2246,  0.0651,
           0.1079,  0.0919,  0.0621]]]) A
tensor([[[-0.0475,  0.1471, -0.4328,  1.1235, -0.2296,  0.0794, -0.4140,
           0.7596,  0.1517,  0.4127]]]) G
tensor([[[-0.0756,  0.0485,  0.0739,  1.1230,  0.1303,  0.1425,  0.0573,
           0.5451,  0.0640,  0.2182]]]) E
tensor([[[ 0.0868,  0.0483, -0.0029,  0.8183, -0.0898,  0.4581, -0.2307,
           0.3969, -0.0639,  0.3571]]]) D
tensor([[[ 0.5681,  0.0087, -0.3517,  0.4695, -0.3098,  0.5262,  0.0381,
           0.6060, -0.2616,  0.3999]]]) F
tensor([[[ 0.3121,  0.2657, -0.3490,  1.0029, -0.2813,  0.2085, -0.2208,
           1.2155, -0.3250,  0.4873]]]) G
tensor([[[-0.3003,  0.2418, -0.1286,  1.0362, -0.1871,  0.1969, -0.2031,
           1.2564, -0.1685,  0.1829]]]) A
tensor([[[ 0.0153,  0.0784, -0.1125,  1.0010, -0.2211,  0.4120, -0.2861,
           0.6271, -0.1485,  0.5295]]]) C
tensor([[[ 0.3555, -0.0468, -0.5395,  0.7708, -0.4197,  0.2116, -0.2613,
       

In [69]:
total_samples = 20
l = len(c)
D = np.zeros((total_samples, total_samples), dtype=float)

for ii in range(total_samples):
    for jj in range(total_samples):
        D[ii,jj] = cosine(c[l-ii-1][0][0], c[l-jj-1][0][0]) 
        print(data[l-ii-1], data[l-jj-1], D[ii,jj])

D D 0.0
D F 0.15633351143560537
D G 0.13484026699799057
D A 0.33571905129269763
D C 0.10598186178616587
D B 0.11213971219103624
D G 0.07864204260555008
D D 0.10859151257157962
D F 0.1402677014965369
D E 0.09946345642790455
D G 0.15527930283731384
D C 0.29570605617252466
D A 0.17880870153770323
D B 0.1216979905985428
D G 0.09800057478724367
D D 0.12507156248013462
D F 0.10992496139691488
D E 0.10062371441914741
D G 0.10939884984008696
D D 0.2395507499524442
F D 0.15633351143560537
F F 1.7823715392495387e-08
F G 0.13315009893902352
F A 0.45686464940953075
F C 0.22258837084022098
F B 0.15121831764353066
F G 0.13771615012381644
F D 0.34683575699398494
F F 0.1716213705546853
F E 0.1400995342831728
F G 0.19198929121599229
F C 0.42816334173521775
F A 0.2190342471989375
F B 0.15311163360643942
F G 0.17699992337101478
F D 0.37182647591074536
F F 0.12020179571785683
F E 0.09380068419641374
F G 0.1600334127986106
F D 0.5191707858363372
G D 0.13484026699799057
G F 0.13315009893902352
G G 0.0
G A 0

In [48]:
D

array([[2.32856496e-08, 2.07265260e-03, 1.57299895e-01, 1.92165015e-01,
        3.14233280e-02, 3.17906631e-02, 1.08587563e-01, 2.53117544e-01],
       [2.07265260e-03, 0.00000000e+00, 1.88830848e-01, 2.12786373e-01,
        4.22840054e-02, 3.99942949e-02, 1.34117044e-01, 2.56191782e-01],
       [1.57299895e-01, 1.88830848e-01, 2.11798856e-09, 1.69117273e-01,
        1.31069536e-01, 1.33580997e-01, 2.60504235e-02, 2.52264275e-01],
       [1.92165015e-01, 2.12786373e-01, 1.69117273e-01, 0.00000000e+00,
        1.16378974e-01, 2.78109323e-01, 1.05473124e-01, 1.06674866e-01],
       [3.14233280e-02, 4.22840054e-02, 1.31069536e-01, 1.16378974e-01,
        1.58367839e-08, 4.78101340e-02, 5.62914780e-02, 2.01744972e-01],
       [3.17906631e-02, 3.99942949e-02, 1.33580997e-01, 2.78109323e-01,
        4.78101340e-02, 4.95708463e-09, 9.55245821e-02, 3.36495603e-01],
       [1.08587563e-01, 1.34117044e-01, 2.60504235e-02, 1.05473124e-01,
        5.62914780e-02, 9.55245821e-02, 0.00000000e+00, 1.

In [37]:
data[l-8:l]

'DGBCAGFD'

In [38]:
c[l-ii-1][0][0]

tensor([-0.7345, -0.5629,  1.0458, -0.4893,  0.4876])